# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Microsoft Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.


## راه‌اندازی

قبل از اجرای این نوت‌بوک، مطمئن شوید که:

1. **یک پروژه Microsoft Foundry** با یک مدل چت مستقر شده (مثلاً `gpt-5-mini`) دارید.
2. **وارد Azure CLI شده‌اید** — در ترمینال خود دستور `az login` را اجرا کنید.
3. **متغیرهای محیطی مورد نیاز را تنظیم کرده‌اید:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطه پایانی پروژه Microsoft Foundry شما.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.

سلول زیر بسته‌های پایتون مورد نیاز شما را نصب می‌کند.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ایجاد اولین عامل شما

یک عامل به دو چیز نیاز دارد:

- **دستورالعمل‌ها** که به آن می‌گویند *که کیست* و *چگونه رفتار کند* (یک پیام سیستم).
- **ابزارها** — توابع پایتون که با `@tool` تزئین شده‌اند و عامل می‌تواند برای دریافت اطلاعات یا انجام کارها فراخوانی کند.

در ادامه یک ابزار ساده تعریف می‌کنیم که فهرستی از مقاصد محبوب تعطیلات را برمی‌گرداند. عامل از این ابزار زمانی استفاده خواهد کرد که کاربر درخواست پیشنهادات سفر کند.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## پاسخ‌های جریانی

برای تجربه‌ای تعاملی‌تر می‌توانید پاسخ عامل را به صورت **جریانی** دریافت کنید. به جای صبر کردن برای پاسخ کامل، عامل بخش‌های متنی را به محض تولید شدن ارائه می‌دهد. این روش به‌ویژه در رابط‌های گفت‌وگو مفید است، جایی که می‌خواهید خروجی را به صورت زمان واقعی نمایش دهید.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصه

در این درس آموختید چگونه:

- **یک تامین‌کننده ایجاد کنید** که از طریق `FoundryChatClient` به سرویس Microsoft Foundry Agent متصل می‌شود.
- **یک ابزار تعریف کنید** با استفاده از دکوراتور `@tool` تا عامل بتواند توابع پایتون شما را فراخوانی کند.
- **عامل را اجرا کنید** با یک پیام کاربر و پاسخ آن را چاپ کنید.
- **پاسخ‌ها را به صورت زنده پخش کنید** برای خروجی بلادرنگ.

در درس بعد، چارچوب‌های عاملی را عمیق‌تر بررسی خواهیم کرد و یاد می‌گیریم چگونه ابزارهای قدرتمندتر و توانایی‌های استدلال چندمرحله‌ای را به عوامل بدهیم.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
